# Azure OpenAI to OpenAI on Amazon Bedrock Validation Notebook

This notebook is the executable validation bench for the migration cookbook. It does not provision Azure or AWS resources. It validates the checked-in cookbook examples, compares sanitized Azure and AWS golden-request evidence, and keeps live Bedrock invocation behind an explicit opt-in flag.

## Goal

Use this notebook to prove the migration pattern preserves business behavior:

- Local cookbook checks pass.
- Python migration snippets parse and compile.
- The golden Azure and AWS responses match on deterministic workflow outcomes.
- Narrative model text may differ without failing validation.
- Live Bedrock smoke testing runs only when `RUN_LIVE_BEDROCK=1`.

## Setup

The notebook uses only checked-in files for the default path. Optional live Bedrock testing requires AWS credentials, access to the selected model in the selected region, and Bedrock Mantle token configuration.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Could not find repository root")


repo_root = find_repo_root(Path.cwd().resolve())
examples_dir = repo_root / "examples"
python_examples_dir = examples_dir / "python"
sys.path.insert(0, str(python_examples_dir))

run_live_bedrock = os.getenv("RUN_LIVE_BEDROCK") == "1"
print(f"Repository root: {repo_root}")
print(f"RUN_LIVE_BEDROCK={int(run_live_bedrock)}")

## 1. Run local release checks

`npm test` runs the public-safety scan, validates source formatting, parses Python examples, and builds the single-page site. The Python compile step mirrors the CI gate that checks every example file.

In [ ]:
def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(args, cwd=repo_root, text=True, capture_output=True)
    print("$ " + " ".join(args))
    if result.stdout:
        print("\n".join(result.stdout.splitlines()[-30:]))
    if result.stderr:
        print("\n".join(result.stderr.splitlines()[-20:]))
    result.check_returncode()
    return result


run_command(["npm", "test"])
python_files = [str(path) for path in sorted(python_examples_dir.glob("*.py"))]
run_command([sys.executable, "-m", "py_compile", *python_files])

## 2. Load the golden request and evidence fixtures

The fixtures are sanitized examples. They are not live customer evidence and they do not contain account IDs, tenant IDs, endpoints, or secrets.

In [ ]:
golden_request = json.loads((examples_dir / "golden-request.json").read_text())
azure_output = json.loads((examples_dir / "evidence" / "azure-golden-request-response.example.json").read_text())
aws_output = json.loads((examples_dir / "evidence" / "aws-golden-request-response.example.json").read_text())

print("Golden request:")
print(json.dumps(golden_request, indent=2))

## 3. Compare deterministic migration outcomes

The comparison allows generated prose and citation relevance scores to differ. It fails when workflow state, risk level, required approvals, deterministic findings, or citation structure drift.

In [ ]:
from compare_golden_outputs import assert_equal_operational_outcome, normalize_list


assert_equal_operational_outcome(azure_output, aws_output)

comparison_rows = [
    ("workflow state", azure_output["status"], aws_output["status"]),
    ("risk level", azure_output["risk_level"], aws_output["risk_level"]),
    (
        "required approvals",
        ", ".join(normalize_list(azure_output["required_approvals"])),
        ", ".join(normalize_list(aws_output["required_approvals"])),
    ),
    (
        "deterministic findings",
        ", ".join(azure_output["analysis"]["deterministic_findings"]),
        ", ".join(aws_output["analysis"]["deterministic_findings"]),
    ),
]

for label, azure_value, aws_value in comparison_rows:
    print(f"{label}: Azure={azure_value} | AWS={aws_value}")

print("\nOperational outcome: MATCH")

## 4. Check citation and audit evidence

A migrated workflow needs evidence and auditability, not just matching status labels.

In [ ]:
required_audit_events = {
    "request_submitted",
    "rules_evaluated",
    "workflow_status_changed",
    "policy_retrieval_completed",
    "llm_analysis_generated",
    "escalation_email_drafted",
}

for provider_name, output in [("azure", azure_output), ("aws", aws_output)]:
    citations = output["analysis"]["policy_citations"]
    audit_events = set(output["audit_events"])
    missing_events = required_audit_events - audit_events
    assert citations, f"{provider_name} citations are empty"
    assert all(citation.get("source_document") for citation in citations)
    assert all(citation.get("section_title") for citation in citations)
    assert not missing_events, f"{provider_name} missing audit events: {sorted(missing_events)}"
    print(f"{provider_name}: {len(citations)} citations, {len(audit_events)} audit events")

print("Citation and audit checks: PASS")

## 5. Optional live Bedrock Mantle smoke

This cell is skipped unless `RUN_LIVE_BEDROCK=1`. A live pass proves the configured AWS identity, region, token, endpoint, and model can answer a minimal Responses API request. It does not prove the whole migration by itself.

In [ ]:
if not run_live_bedrock:
    print("Skipped live Bedrock smoke. Set RUN_LIVE_BEDROCK=1 to opt in.")
else:
    from openai import BedrockOpenAI

    aws_region = os.environ["AWS_REGION"]
    model_id = os.getenv("AWS_BEDROCK_MODEL_ID", "openai.gpt-5.4")
    client = BedrockOpenAI(aws_region=aws_region)
    response = client.responses.create(
        model=model_id,
        instructions="Return one concise sentence about migration validation.",
        input="Confirm this Bedrock Mantle smoke request was received.",
        max_output_tokens=80,
    )
    print(response.output_text[:500])

## What the checks establish

- The public cookbook source passes local release checks.
- The Python examples compile.
- The golden request preserves deterministic operational outcomes across Azure-shaped and AWS-shaped evidence.
- Citations and audit events are present in both outputs.
- Live Bedrock validation remains explicit and opt-in.

Do not describe live AWS or Bedrock as validated unless the optional live cell was executed with real credentials and captured as review evidence.